---
authors:
  - name: Ignacio Ibarra
  - name: Lukas Heumos
    roles:
      - writing – review & editing
  - name: Anna Schaar
    roles:
      - writing – review & editing
---


(chromatin-accessibility-gene-regulatory-networks)=
# Gene regulatory networks
```{dropdown} 🧠 Key takeaways

:::{card}
:link: #chromatin-accessibility-gene-regulatory-networks-atac-key-takeaway-1
Multiome data allow regions to be linked to genes and TFs to regions in the same cells, but the resulting networks are hypotheses that differ between methods.
:::

:::{card}
:link: #chromatin-accessibility-gene-regulatory-networks-atac-key-takeaway-2
Region-gene links combine genomic proximity with the correlation of accessibility and expression, and TF-region links require a binding motif.
:::

:::{card}
:link: #chromatin-accessibility-gene-regulatory-networks-atac-key-takeaway-3
Only a small fraction of inferred links is supported by prior knowledge, so networks need validation, ideally by perturbing the TFs.
:::

```
``````{dropdown} ⚙️ Environment setup
`````{tab-set}

````{tab-item} Steps
```{include} ../_static/default_text_env_setup.md
```
````

````{tab-item} yml
```{literalinclude} chromatin_accessibility.yml
:language: yaml
```
````

`````
``````
````{dropdown} 🗄️ Get data and notebooks
```{include} ../_static/default_text_lamindb_setup.md
```
````
<!-- END DROPDOWNS -->


(chromatin-accessibility-gene-regulatory-networks-atac-key-takeaway-1)=
## Motivation

Transcription factors (TFs) regulate genes by binding to accessible regulatory regions, such as promoters and enhancers, whose accessibility changes with their activity {cite:p}`atac:Spitz2012-sw`.
Multiome data measure accessibility and expression in the same cells, which allows us to link regions to the genes whose expression follows their accessibility, and TFs to the regions that contain their motifs.
Together, these links form a network from TFs through regions to target genes, the idea behind methods such as SCENIC+ {cite:p}`atacgrn:bravogonzalezblas2023`.

Such networks are hypotheses rather than measurements: the networks of different methods barely overlap, change with random seeds and preprocessing, and predict the effects of perturbations poorly {cite:p}`atacgrn:badiaimompel2024greta`.
For TF activities from gene expression alone, see the [gene regulatory network chapter](../mechanisms/gene_regulatory_networks.ipynb) of the RNA section.

In [1]:
import lamindb as ln
import pandas as pd
import scanpy as sc
import snapatac2 as snap

ln.connect("theislab/sc-best-practices")

ln.track("N64Xsy5YJqH4")

→ connected lamindb: theislab/sc-best-practices


→ loaded Transform('N64Xsy5YJqH40000', key='gene_regulatory_networks_atac.ipynb'), started new Run('Zh98aN728L2pCIMK') at 2026-09-29 15:44:48 UTC


→ notebook imports: lamindb-core==2.10.0 pandas==2.3.3 scanpy==1.12.4 snapatac2==2.10.0


We load the peak matrix from the [previous chapter](peaks_motifs.ipynb) and the gene expression of the same cells, and normalize both with the shifted logarithm.
We restrict the network to the cell type-specific peaks and to the 2,000 most variable genes, which keeps it small and focused on regulation that differs between cell types.

In [2]:
peak_matrix = ln.Artifact.get(key="chromatin_accessibility/pbmc10k_peaks.h5ad").load()
rna = ln.Artifact.get(
    key="introduction/advanced_data_structures_and_frameworks_muon.h5mu"
).load()["rna"]
rna = rna[peak_matrix.obs_names].copy()
rna.var_names_make_unique()
for data in (peak_matrix, rna):
    sc.pp.normalize_total(data)
    sc.pp.log1p(data)
sc.pp.highly_variable_genes(rna, n_top_genes=2000)
markers = snap.tl.marker_regions(peak_matrix, groupby="cell_type", pvalue=0.01)
regions = sorted({region for peaks in markers.values() for region in peaks})

(chromatin-accessibility-gene-regulatory-networks-atac-key-takeaway-2)=
## Linking regions to genes

We connect every region to the genes whose transcription start site lies within 250 kb and keep the links in which accessibility and expression are correlated across cells.

In [3]:
annotation = ln.Artifact.get(
    key="chromatin_accessibility/gencode_v41_GRCh38.gff3.gz"
).cache()
network = snap.tl.init_network_from_annotation(regions, annotation)
variable_genes = set(rna.var_names[rna.var["highly_variable"]])
network = snap.tl.prune_network(
    network, node_filter=lambda node: node.type != "gene" or node.id in variable_genes
)
snap.tl.add_cor_scores(network, gene_mat=rna, peak_mat=peak_matrix)
network = snap.tl.prune_network(
    network, edge_filter=lambda source, target, link: link.cor_score > 0.1
)

2026-09-29 17:45:03 - INFO - Removed 22374 isolated nodes.


  0%|          | 0/1204 [00:00<?, ?it/s]

  4%|▎         | 44/1204 [00:00<00:02, 432.77it/s]

  7%|▋         | 89/1204 [00:00<00:02, 436.39it/s]

 11%|█         | 133/1204 [00:00<00:02, 412.37it/s]

...

 92%|█████████▏| 1108/1204 [00:02<00:00, 420.65it/s]

 96%|█████████▌| 1152/1204 [00:02<00:00, 425.19it/s]

 99%|█████████▉| 1195/1204 [00:02<00:00, 426.13it/s]

100%|██████████| 1204/1204 [00:02<00:00, 424.78it/s]


2026-09-29 17:45:06 - INFO - Removed 10238 isolated nodes.


In [4]:
f"{network.num_edges()} region-gene links"

'541 region-gene links'

## Adding transcription factors

Next, we connect TFs to the linked regions that contain their CIS-BP motif {cite:p}`atacgrn:weirauch2014` and derive TF-gene links through these regions {cite:p}`atacgrn:zhang2024snapatac2`.
A motif only shows that a TF could bind, so we additionally require that the TF and its target are co-expressed, and we remove links of TFs to their own gene, whose correlation is trivially 1.

In [5]:
fasta = ln.Artifact.get(key="chromatin_accessibility/gencode_v41_GRCh38.fa").cache()
motif_file = ln.Artifact.get(key="chromatin_accessibility/cisBP_human.meme").cache()
cis_bp = {}
for motif in snap.read_motifs(str(motif_file)):
    motif.name = motif.id.split("+")[0]
    if (
        motif.name not in cis_bp
        or motif.info_content() > cis_bp[motif.name].info_content()
    ):
        cis_bp[motif.name] = motif
snap.tl.add_tf_binding(network, motifs=list(cis_bp.values()), genome_fasta=fasta)
grn = snap.tl.link_tf_to_gene(network)
snap.tl.add_cor_scores(grn, gene_mat=rna, peak_mat=peak_matrix)
grn = snap.tl.prune_network(
    grn,
    edge_filter=lambda source, target, link: source != target and link.cor_score > 0.1,
)
edges = pd.DataFrame(
    [
        (grn[source].id, grn[target].id, grn.get_edge_data(source, target).cor_score)
        for source, target in grn.edge_list()
    ],
    columns=["tf", "target", "correlation"],
)

2026-09-29 17:45:07 - INFO - Fetching 524 sequences ...


2026-09-29 17:45:07 - INFO - Searching for the binding sites of 1165 motifs ...


  0%|          | 0/1165 [00:00<?, ?it/s]

  1%|          | 7/1165 [00:00<00:18, 62.18it/s]

  2%|▏         | 24/1165 [00:00<00:09, 121.47it/s]

...

100%|██████████| 1165/1165 [00:17<00:00, 67.46it/s]

  0%|          | 0/156 [00:00<?, ?it/s]

 99%|█████████▊| 154/156 [00:00<00:00, 1533.81it/s]

100%|██████████| 156/156 [00:00<00:00, 1518.02it/s]


2026-09-29 17:45:24 - INFO - Removed 67 isolated nodes.


In [6]:
edges["tf"].value_counts().head(10)

tf
BCL11A     30
MEF2C      23
KLF4       11
POU2AF1    11
IRF8        8
IRF4        4
YBX3        3
EBF1        3
TCF4        3
CREB5       2
Name: count, dtype: int64

The network consists of 115 links from 23 TFs to 84 genes, most of them from regulators of B cells, pDCs and monocytes such as BCL11A, MEF2C, POU2AF1, IRF8 and KLF4.
`link_tf_to_gene` only keeps TFs whose own gene is part of the network, so regulators that are not among the highly variable genes, such as LEF1, TCF7, CEBPA and SPI1, are missing although their motifs are enriched in the [previous chapter](peaks_motifs.ipynb).

(chromatin-accessibility-gene-regulatory-networks-atac-key-takeaway-3)=
## Comparison with prior knowledge

To see how much of the network is supported by known regulation, we compare its TF-gene links with CollecTRI {cite:p}`atacgrn:mullerdott2023`, the prior-knowledge network that we use in the RNA section.

In [7]:
collectri = ln.Artifact.get(key="conditions/gsea_pathway/collectri.parquet").load()
known = set(zip(collectri["source"], collectri["target"], strict=True))
edges["in_collectri"] = [
    edge in known for edge in zip(edges["tf"], edges["target"], strict=True)
]
edges[edges["in_collectri"]]

,tf,target,correlation,in_collectri
7,TCF7L2,PLAUR,0.418005,True
59,KLF4,CDKN1C,0.144577,True
64,KLF4,IFITM3,0.261776,True
98,BCL11A,TCF4,0.379709,True


Only 4 of the 115 links are in CollecTRI, although CollecTRI contains all 23 TFs.
Prior knowledge is incomplete and not specific to cell types, so the other links are not necessarily wrong, but they remain hypotheses until they are tested, for example by perturbing the TF.

## Quiz

### Theory

In [8]:
%run ../../scripts/quiz.py

with quiz_tabs():
    flip_card(
        "q1",
        "Are there generally more chromatin peaks than genes when processing ATAC and RNA data? What is the consequence at the level of structure, when building GRNs?",
        "Generally, there are more chromatin peaks than genes in ATAC-seq data, leading to a many-to-one relationship in gene regulatory networks (GRNs) where multiple regulatory elements can control a single gene.",
        front_font_size=15,
        back_font_size=15,
    )
    flip_card(
        "q2",
        "What is considered a TF activator/repressor at the level of chromatin accessibility, and at the level of gene regulation?",
        "At the level of chromatin accessibility, transcription factor (TF) activators increase accessibility by promoting an open chromatin state, while repressors decrease accessibility by inducing a closed state; in terms of gene regulation, activators enhance transcription, whereas repressors inhibit it.",
        back_font_size=13,
    )
    flip_card(
        "q3",
        "Which additional readouts are complementary to scRNA-seq and scATAC-seq, when interpreting ATAC+RNA GRN models?",
        "Complementary readouts to scRNA-seq and scATAC-seq include techniques like ChIP-seq for profiling protein-DNA interactions and Hi-C for capturing chromatin conformation, providing additional layers of regulatory information.",
        back_font_size=15,
    )

### Networks from multiome data

In [9]:
%run ../../scripts/quiz.py

with quiz_tabs():
    flip_card(
        "q4",
        "Why is a TF motif in an accessible region not enough evidence that the TF regulates a nearby gene?",
        "A motif only shows that the TF could bind. Many motif instances are never bound, TFs of the same family share motifs, and the region may regulate a different gene or none at all.",
        back_font_size=15,
    )
    flip_card(
        "q5",
        "Why do multiome data help to link regions to genes?",
        "Accessibility and expression are measured in the same cells, so a region whose accessibility follows the expression of a nearby gene across cells can be linked to it without matching cells between separate experiments.",
        back_font_size=15,
    )
    flip_card(
        "q6",
        "How can the links of an inferred network be validated?",
        "By comparing them with prior knowledge such as curated TF-target interactions, with TF binding measured by ChIP-seq, and with the effects of perturbing the TF, which is the strongest test of a causal link.",
        back_font_size=15,
    )

## Contributors
We gratefully acknowledge the contributions of:
### Authors
* Ignacio Ibarra
### Reviewers
* Lukas Heumos
* Anna Schaar